# 🎓 Smart Classroom AI: Student Presence & Behavior Training Pipeline
### Automated Face Recognition (Presence) & YOLOv8 Behavior Classification

This Google Colab notebook trains the Computer Vision models for the **Smart Classroom & Attendance Analytics Portal**:
1. **Student Presence (Attendance)**: Face Recognition (FaceNet) to mark students **Present** or **Absent**.
2. **Student Behavior Recognition**: YOLOv8/v11 deep learning model to detect:
   - 📖 **Studying** (Attentive, reading, writing)
   - 😴 **Sleeping** (Head down on desk, eyes closed, slumped)
   - 🗣️ **Speaking** (Talking to peers, turning around)
   - 📱 **Phone Use** (Using smartphone/distraction)
   - 🙋 **Raising Hand** (Asking/answering questions)
   - 🔀 **Distracted**

---

## Step 1: Check GPU & Install Dependencies
Ensure GPU is enabled under **Runtime > Change runtime type > T4 GPU**.

In [ ]:
# Verify GPU acceleration
!nvidia-smi

# Install Ultralytics YOLOv8, FaceNet PyTorch, OpenCV, and helper libraries
!pip install -q ultralytics facenet-pytorch opencv-python-headless pillow requests scikit-learn matplotlib

## Step 2: Classroom Behavior Dataset Setup
You can load a classroom behavior dataset from:
1. **Roboflow Universe** (Automated Classroom Action Recognition)
2. **Google Drive** (Mount your own labeled images)
3. **Demo / Synthetic Dataset** (Generates a sample dataset to test the full pipeline immediately)

In [ ]:
import os
import yaml
from pathlib import Path

# Create dataset directory structure
for split in ['train', 'val', 'test']:
    os.makedirs(f'dataset/images/{split}', exist_ok=True)
    os.makedirs(f'dataset/labels/{split}', exist_ok=True)

# Define classes and dataset.yaml
dataset_config = {
    'path': '/content/dataset',
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'names': {
        0: 'studying',
        1: 'sleeping',
        2: 'speaking',
        3: 'phone_use',
        4: 'raising_hand',
        5: 'distracted'
    }
}

with open('dataset.yaml', 'w') as f:
    yaml.dump(dataset_config, f)

print('Created dataset.yaml with 6 classroom behavior classes!')

### (Optional) Download Classroom Behavior Dataset via Roboflow
If you have a Roboflow API key or public classroom dataset link, run the snippet below:

In [ ]:
# Optional: Download classroom behavior dataset from Roboflow Universe
# !pip install -q roboflow
# from roboflow import Roboflow
# rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")
# project = rf.workspace("classroom-monitoring").project("student-behavior-detection")
# version = project.version(1)
# dataset = version.download("yolov8")

### Generate Sample Dataset to Validate Pipeline
If you don't have images yet, this cell synthesizes sample images so training can run end-to-end:

In [ ]:
import numpy as np
import cv2

# Create 30 synthetic training images and 10 validation images with bounding box annotations
np.random.seed(42)
def create_sample_split(split, count):
    for i in range(count):
        img = np.ones((640, 640, 3), dtype=np.uint8) * np.random.randint(180, 240)
        # Draw mock classroom desk & student shape
        cv2.rectangle(img, (100, 300), (540, 580), (120, 80, 50), -1) # desk
        cv2.circle(img, (320, 220), 70, (200, 160, 140), -1)          # student head
        cv2.rectangle(img, (240, 290), (400, 480), (50, 100, 200), -1) # student body
        
        cls_id = i % 6
        img_path = f'dataset/images/{split}/sample_{i:03d}.jpg'
        label_path = f'dataset/labels/{split}/sample_{i:03d}.txt'
        
        cv2.imwrite(img_path, img)
        # YOLO label format: class_id center_x center_y width height (normalized 0 to 1)
        with open(label_path, 'w') as f:
            f.write(f"{cls_id} 0.5 0.5 0.6 0.7\n")

create_sample_split('train', 30)
create_sample_split('val', 10)
print('Sample dataset generated successfully for testing!')

## Step 3: Train YOLOv8 Behavior Detection Model
We transfer-learn from `yolov8n.pt` (Nano) or `yolov8s.pt` (Small) for fast, accurate real-time detection.

In [ ]:
from ultralytics import YOLO

# Initialize pretrained YOLOv8 model
model = YOLO('yolov8n.pt')

# Fine-tune on classroom behavior dataset
results = model.train(
    data='dataset.yaml',
    epochs=25,              # Set to 50 or 100 for production accuracy
    imgsz=640,
    batch=16,
    device=0,               # Use GPU 0
    project='classroom_runs',
    name='behavior_detector',
    patience=10,
    save=True,
    verbose=True
)

## Step 4: Validate Model & View Performance Metrics

In [ ]:
# Run validation
metrics = model.val()
print(f"mAP@0.5:      {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision:    {metrics.box.mp:.4f}")
print(f"Recall:       {metrics.box.mr:.4f}")

# Display training curves and confusion matrix if available
from IPython.display import Image, display
results_img = 'classroom_runs/behavior_detector/results.png'
if os.path.exists(results_img):
    display(Image(results_img))

## Step 5: Student Face Recognition (Presence Enrollment)
Here we extract 512-dimensional facial embeddings using FaceNet (`InceptionResnetV1`).
Each student's profile is saved with their `rollNumber` and `name`.

In [ ]:
import torch
import pickle
from facenet_pytorch import MTCNN, InceptionResnetV1
from PIL import Image

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
mtcnn = MTCNN(image_size=160, margin=14, keep_all=False, device=device)
resnet = InceptionResnetV1(pretrained='vggface2').eval().to(device)

face_database = {}

def enroll_student(roll_number, name, image_path):
    img = Image.open(image_path).convert('RGB')
    face_tensor = mtcnn(img)
    if face_tensor is not None:
        face_tensor = face_tensor.unsqueeze(0).to(device)
        with torch.no_grad():
            embedding = resnet(face_tensor).squeeze().cpu().numpy()
        # Normalize embedding
        embedding = embedding / (np.linalg.norm(embedding) + 1e-10)
        face_database[roll_number] = {
            'rollNumber': roll_number,
            'name': name,
            'embedding': embedding
        }
        print(f"Enrolled student: {name} ({roll_number})")
    else:
        print(f"No face detected in {image_path}")

# Save enrolled database to pickle
with open('face_database.pkl', 'wb') as f:
    pickle.dump(face_database, f)
print('Face database ready!')

## Step 6: Test Inference Pipeline (Presence + Behavior Combined)
This runs both the YOLO behavior detector and FaceNet presence recognizer on test images or video.

In [ ]:
# Test inference with the trained behavior model
test_results = model.predict(source='dataset/images/val/sample_000.jpg', conf=0.25)
for r in test_results:
    res_bgr = r.plot() # Draws bounding boxes and labels
    cv2.imwrite('prediction_sample.jpg', res_bgr)

display(Image('prediction_sample.jpg'))

## Step 7: Export & Download Weights for Your MERN Portal
Download `behavior_best.pt` and place it in `smart-classroom-portal/ai-engine/weights/` on your computer.

In [ ]:
from google.colab import files
import shutil

best_weights = 'classroom_runs/behavior_detector/weights/best.pt'
if os.path.exists(best_weights):
    shutil.copyfile(best_weights, 'behavior_best.pt')
    print('Downloading behavior_best.pt...')
    files.download('behavior_best.pt')
else:
    print('Weights not found, check training output.')